# HW11: find one scanline in a distorted copy
Use **HW11_pyramid_A.png** (source) and **HW11_pyramid_B.png** (target) for both tasks. Download them from the HW11 page or Drive and put them beside this notebook (in Colab, upload them to the runtime's Files panel). With both PNGs present, no archive or network connection is needed. If neither PNG is present, the setup regenerates and saves the same pair from `pyramid1.png` in the original course ZIP.

T4: compare pointwise distance with unrestricted DTW. T5: keep both images and the query fixed and change only the DTW band. You implement the scoring, ranking and traceback; the helpers load images and plot your results.

In [ ]:
from hashlib import sha256
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile, is_zipfile

PAGE = "https://courses.cs.ut.ee/2026/algorithmics/fall/Main/HW11"

def course_zip(name):
    path = Path(name)
    if not path.is_file():
        url = f"{PAGE}?action=download&upname={name}"
        try:
            with urlopen(url, timeout=45) as response:
                data = response.read()
        except OSError as error:
            raise RuntimeError(f"Download {url} and place {name} beside this notebook.") from error
        if not is_zipfile(BytesIO(data)):
            raise ValueError(f"Expected a ZIP from {url}; download it manually.")
        path.write_bytes(data)
    return ZipFile(path)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

def image_panel(ax, pixels, title):
    ax.imshow(pixels, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set(title=title, xlabel="Column x", ylabel="Row y (0 at top)")

In [ ]:
PAIR_NAMES = ("HW11_pyramid_A.png", "HW11_pyramid_B.png")
PIXEL_HASHES = ("1c604093f348be71fa322c22ce628b041530fe4284801cf327619837ff538536",
                "217be88cf051705da12b919dfee616b8fe9908497ac56950bfe9f55ce2356fed")
SHIFT_ROWS = 8

def make_target(a):
    """Squeeze the middle of A, shift down eight rows, and round to PNG brightness."""
    height, width = a.shape
    x = np.arange(width)
    source_knots = (width - 1) * np.array([0, 0.25, 0.70, 1])
    target_knots = (width - 1) * np.array([0, 0.40, 0.58, 1])
    sample_at = np.interp(x, target_knots, source_knots)
    warped = np.stack([np.interp(sample_at, x, row) for row in a])
    source_y = np.maximum(np.arange(height) - SHIFT_ROWS, 0)  # Repeat top edge; no wrap.
    return np.rint(warped[source_y]).astype(np.uint8)

def make_pair(original):
    a = np.asarray(original.convert("L").resize((150, 100), Image.Resampling.BOX))
    return a, make_target(a)

def check_pixels(pixels, name, expected):
    if (pixels.shape != (100, 150) or pixels.dtype != np.uint8
            or sha256(pixels.tobytes()).hexdigest() != expected):
        raise ValueError(f"{name} differs from the course input. Use the unchanged grayscale PNG.")

def load_pair():
    paths = [Path(name) for name in PAIR_NAMES]
    present = [p.is_file() for p in paths]
    if any(present) and not all(present):
        raise FileNotFoundError(f"Place both PNGs in the working directory; missing {paths[present.index(False)]}.")
    if all(present):
        pair = []
        for path in paths:
            with Image.open(path) as image:
                if image.format != "PNG" or image.mode != "L":
                    raise ValueError(f"{path}: expected a grayscale PNG, not a screenshot or RGB conversion.")
                pair.append(np.array(image))
    else:
        with course_zip("HW11images.zip") as archive, archive.open("pyramid1.png") as file:
            with Image.open(file) as original:
                pair = make_pair(original)
    for pixels, path, expected in zip(pair, paths, PIXEL_HASHES):
        check_pixels(pixels, path.name, expected)
    if not any(present):
        for pixels, path in zip(pair, paths):
            Image.fromarray(pixels).save(path)
    print(("Loaded" if all(present) else "Regenerated"), " and ".join(PAIR_NAMES))
    return tuple(pixels.astype(float) for pixels in pair)  # Avoid unsigned subtraction.

## T4. One query against 100 rows
A and B are already grayscale at the required size. Do not resize them again. The query is row 55 of A, read left to right. In B the pyramid has been squeezed horizontally and the scene shifted down. Score the entire query against each row of B by brightness alone. Use the known transformation to check your ranking **after** the search.

In [ ]:
A, B = load_pair()
QUERY_Y = 55
query = A[QUERY_Y].copy()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
image_panel(axes[0], A, "A: source")
axes[0].axhline(QUERY_Y, color="#b21a60", linewidth=1.8)
image_panel(axes[1], B, "B: search all 100 target rows")
plt.show()
print(f"Query: A[{QUERY_Y}], length {len(query)}. Candidates: {B.shape[0]} rows.")

### Turn your results into a report figure
Pass two arrays of 100 scores in **candidate row order**, not sorted by score. Also pass your two winning row numbers. The figure shows the marked images, the complete search and the selected profiles. The display helper never computes either score or picks a winner.

In [ ]:
def show_row_search(a, b, query_y, point_scores, dtw_scores, point_y, dtw_y):
    colors = ("#bd6413", "#147486")
    methods = ("Pointwise", "DTW")
    scores = np.asarray([point_scores, dtw_scores], dtype=float)
    if scores.shape != (2, len(b)) or not np.isfinite(scores).all() or np.any(scores < 0):
        raise ValueError("Pass one nonnegative finite score per candidate row and method.")
    if not (0 <= query_y < len(a) and 0 <= point_y < len(b) and 0 <= dtw_y < len(b)):
        raise ValueError("Row numbers must be indices in the displayed images.")

    fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
    left, right = axes[0]
    image_panel(left, a, f"A: query row {query_y}")
    left.axhline(query_y, color="#b21a60", linewidth=2)
    image_panel(right, b, f"B: pointwise row {point_y} / DTW row {dtw_y}")
    cost_ax, signal_ax = axes[1]
    signal_ax.plot(a[query_y], color="#b21a60", linewidth=2, label=f"Query {query_y}")
    for method, values, row, color, style in zip(
        methods, scores, (point_y, dtw_y), colors, ("-", "--")
    ):
        right.axhline(row, color=color, linestyle=style, linewidth=2)
        cost_ax.plot(values, color=color, linestyle=style, label=method)
        cost_ax.scatter(row, values[row], color=color, marker="o", s=30)
        signal_ax.plot(b[row], color=color, linestyle=style, label=f"{method} {row}")
    cost_ax.set(title=f"Score of every candidate (n = {len(b)})",
                xlabel="Row in B", ylabel="Total squared cost")
    cost_ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 3))
    signal_ax.set(title="Brightness at equal columns", xlabel="Column x", ylabel="Intensity (0-255)")
    for ax in axes[1]:
        ax.legend(fontsize="small")
    return fig

In [ ]:
# Calculate point_scores[y] and dtw_scores[y] for the query versus B[y].
# Determine point_y and dtw_y yourself; break ties by smaller row number.
# show_row_search(A, B, QUERY_Y, point_scores, dtw_scores, point_y, dtw_y)
# plt.show()

## T5. How wide a band is enough?
Continue with exactly the same A, B and query as T4. B compresses the middle 45% of the image width into 18% and expands the sides. Its vertical shift gives a known counterpart for the query. Rank every row by signal score alone; use the counterpart only to evaluate the results. The guides below mark corresponding columns and rows.

In [ ]:
truth_y = QUERY_Y + SHIFT_ROWS
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
for ax, pixels, title, guides, row in zip(axes, (A, B), ("A: source", "B: squeezed and shifted"),
        ((0.25, 0.70), (0.40, 0.58)), (QUERY_Y, truth_y)):
    image_panel(ax, pixels, title)
    for fraction in guides:
        ax.axvline(fraction * (pixels.shape[1] - 1), color="#147486", linestyle="--")
    ax.axhline(row, color="#b21a60", linewidth=1.5)
plt.show()

### Check a path you reconstructed
Use global DTW with squared local cost; a band restricts columns along the two signals, **not which image rows you search**. For the unrestricted query/counterpart pair, pass the table of accumulated costs without its extra boundary row and column, the forward path of zero-based index pairs, and your score. The helper checks the endpoints, moves and cost and plots the alignment.

In [ ]:
def show_alignment(query, candidate, costs, path, score):
    costs = np.asarray(costs, dtype=float)
    ij = np.asarray(path)
    if costs.shape != (len(query), len(candidate)) or ij.ndim != 2 or ij.shape[1] != 2:
        raise ValueError("Expected one cost per sample pair and a path of index pairs.")
    if not np.issubdtype(ij.dtype, np.integer) or len(ij) == 0:
        raise ValueError("A path needs integer indices and cannot be empty.")
    ends = [(0, 0), (len(query) - 1, len(candidate) - 1)]
    if [tuple(ij[0]), tuple(ij[-1])] != ends:
        raise ValueError("Path must run from the first to the last sample pair.")
    if any(tuple(step) not in {(1, 0), (0, 1), (1, 1)} for step in np.diff(ij, axis=0)):
        raise ValueError("Path contains an invalid step.")
    i, j = ij.T
    if not (np.isclose(np.sum((query[i] - candidate[j]) ** 2), score)
            and np.isclose(costs[-1, -1], score)):
        raise ValueError("Path, reported score and final table cell must agree.")
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
    im = a.imshow(np.ma.masked_invalid(costs), origin="lower", aspect="auto")
    a.plot(j, i, color="red", linewidth=1)
    a.set(xlabel="Candidate index", ylabel="Query index", title="Accumulated cost")
    fig.colorbar(im, ax=a)
    b.plot(query[i], label="query")
    b.plot(candidate[j], label="candidate")
    b.set(xlabel="Step along the path", ylabel="Intensity", title="Aligned signals")
    b.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# Score all 100 rows of B for each band (0, 4, 12, unrestricted).
# Report the winner, rank of truth_y, winning cost and total interior cells evaluated.
# Width 0 must reproduce the T4 pointwise scores on B.
# For the unrestricted pair, call show_alignment(query, B[truth_y], costs, path, score).

## B2. Optional: remove a winding seam
Use image A at its supplied resolution; no additional input is needed. Compare seam removal and ordinary resizing to the same dimensions.

In [ ]:
small = Image.fromarray(A.astype(np.uint8))
ordinary_resize = small.resize((small.width - 20, small.height), Image.Resampling.LANCZOS)
print("Seam experiment:", small.size, "->", ordinary_resize.size)